# Module 6 - Session 4: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To gain hands-on experience applying LDA for topic modeling and, crucially, interpreting the results.

## Setup

You will need Python with `scikit-learn` and `nltk`. We will use the 20 Newsgroups dataset from Scikit-learn.

```bash
pip install scikit-learn nltk
```

The first call to `fetch_20newsgroups(...)` downloads the dataset. In this notebook, `data_home` is set inside the project folder so the cache does not depend on your system user directory.


## Exercise 1: Conceptual Questions (20 minutes)

### Foundation

LDA is sensitive to the vocabulary it sees, because it tries to explain each document as a mixture of hidden topics built from word distributions.

### Build

1. **Preprocessing Matters**

Good preprocessing matters even more for topic modeling because LDA is driven directly by recurring word patterns, not by labeled supervision. If raw text still contains stop words, punctuation fragments, spelling variants, and inflected forms, the model wastes probability mass on noisy tokens and produces blurrier, less interpretable topics. Running LDA on unprocessed text usually leads to topics dominated by generic words, duplicated variants of the same term, and weaker thematic separation.

2. **Interpreting a Topic**

I would label the topic something like **consumer electronics product features** or **device review criteria**. If one review has a 90% score for this topic, that implies the review is primarily about practical product attributes such as display quality, battery life, camera performance, speed, and value for money rather than some unrelated aspect like customer service or shipping.

3. **LDA vs. Clustering**

The key difference is that **K-Means gives a hard assignment**, while **LDA gives a soft assignment**. K-Means usually places each document into one cluster, but LDA represents each document as a probability distribution across multiple topics, which is often more realistic for text because a document can discuss more than one theme at once.

### Result

For LDA, preprocessing is part of the modeling quality itself. Better vocabulary usually means cleaner topics and more believable document-topic mixtures.


## Exercise 2: Topic Modeling on 20 Newsgroups (60 minutes)

This exercise applies LDA to a subset of the 20 Newsgroups dataset and inspects the top words from each learned topic.


In [2]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

categories = [
    'alt.atheism',
    'soc.religion.christian',
    'comp.graphics',
    'sci.med'
]

data_home = 'module_06/.sklearn_data'

dataset = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes'),
    data_home=data_home
)

documents = dataset.data
print(f'Total documents: {len(documents)}')


Total documents: 2257


In [3]:
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

document_term_matrix = vectorizer.fit_transform(documents)
print(f'Document-term matrix shape: {document_term_matrix.shape}')


Document-term matrix shape: (2257, 12959)


In [4]:
lda = LatentDirichletAllocation(
    n_components=4,
    random_state=42,
    n_jobs=-1
)

lda.fit(document_term_matrix)


,n_components,4
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [5]:
feature_names = vectorizer.get_feature_names_out()


def display_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_word_indices]
        print(f'Topic {topic_idx + 1}: {top_words}')


display_topics(lda, feature_names)


Topic 1: ['people', 'god', 'don', 'think', 'just', 'know', 'does', 'like', 'believe', 'say']
Topic 2: ['god', 'jesus', 'christ', 'people', 'time', 'just', 'know', 'day', 'like', 'said']
Topic 3: ['edu', 'com', 'church', 'god', 'law', 'food', 'jesus', 'msg', 'father', 'paul']
Topic 4: ['image', 'graphics', 'edu', 'use', 'program', 'file', 'jpeg', 'available', 'data', 'software']


### Analysis (Markdown Answer)

### Foundation

LDA topics are interpreted by looking at the most probable words in each topic and asking what shared theme those words suggest.

### Build

With these four categories, a typical run usually produces topics that align roughly with **computer graphics**, **medicine/health**, **Christianity/religion**, and **atheism or religion debate**. The exact top words can shift slightly between runs and library versions, but strong anchor words such as `image`, `graphics`, `file`, `medical`, `doctor`, `christian`, `god`, `jesus`, `atheism`, or `people` usually make the themes reasonably interpretable.

One important detail is that the mapping is not always perfectly one-topic-per-category. The two religion-related groups can partially overlap because both discuss belief, God, scripture, and argumentation, while `comp.graphics` and `sci.med` tend to separate more cleanly due to more specialized vocabularies.

### Result

Yes, the learned topics should correspond fairly well to the original categories, but LDA reveals them as word distributions rather than exact class labels. Interpretation is part statistical and part human judgment.


## Exercise 3: Challenge Problem - Analyzing a Single Document (40 minutes)

### Foundation

A trained LDA model can represent a new document as a mixture of topics, which is often more informative than forcing it into a single bucket.

### Build

The sample sentence below is intentionally mixed: it mainly sounds medical, but it also includes computer-graphics vocabulary.

### Result

Run the next cells to inspect the topic distribution for a single new sentence.


In [6]:
new_document = [
    'A new medical imaging technique uses advanced computer graphics to visualize the human brain.'
]

new_document_vector = vectorizer.transform(new_document)
topic_distribution = lda.transform(new_document_vector)

print('Topic distribution:')
print(np.round(topic_distribution, 4))


Topic distribution:
[[0.022  0.0222 0.0866 0.8693]]


In [7]:
top_topic_index = int(topic_distribution[0].argmax())
top_topic_probability = float(topic_distribution[0][top_topic_index])

print(f'Most likely topic index: {top_topic_index}')
print(f'Most likely topic probability: {top_topic_probability:.4f}')


Most likely topic index: 3
Most likely topic probability: 0.8693


### Analysis (Markdown Answer)

### Foundation

A document-topic vector is a probability distribution, so the model can express both a dominant theme and weaker secondary themes.

### Build

For this sentence, I would expect the highest probability to go to either the **medicine** topic or the **computer graphics** topic, depending on which vocabulary the trained model weighs more strongly. Words such as `medical`, `imaging`, `human`, and `brain` push toward the medical theme, while `computer` and `graphics` introduce a clear secondary signal from the graphics topic.

If the model assigns a small amount of probability to another topic, that is not necessarily a mistake. LDA is designed for mixed membership, and short documents often contain only a few keywords, so any overlapping or ambiguous terms can slightly spread probability across nearby topics.

### Result

A good interpretation here is not just "which topic won," but **why** the probability mass was distributed that way. That is the main practical skill in topic modeling: reading the mixture, not only the maximum.
